# Model Comparison Pipeline: EBM, Class Weights, SMOTE, LASSO & Random Forest

## Overview
This notebook compares multiple classification approaches to predict knee replacement outcomes:

1. **Explainable Boosting Machine (EBM)** - An interpretable ML model
2. **Logistic Regression (No Class Weights)** - Baseline model
3. **Logistic Regression (Balanced Class Weights)** - Handles class imbalance via weights
4. **Logistic Regression + SMOTE** - Handles class imbalance via oversampling
5. **LASSO Logistic Regression** - L1 regularization for feature selection
6. **Random forest**

## What We'll Learn
- How to build a unified pipeline to compare multiple models
- How to detect overfitting by comparing train vs validation scores
- How to interpret feature importances
- How to visualize model performance with recall curves
- How cross-validation helps us get reliable performance estimates

## Step 1: Import Libraries

**What's happening here:**
- `pandas` and `numpy`: For data manipulation
- `sklearn`: Machine learning library with models, metrics, and pipelines
- `imblearn`: For SMOTE (handling imbalanced data)
- `interpret`: For the Explainable Boosting Machine
- `matplotlib` and `seaborn`: For visualizations

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')

# Scikit-learn: Models and utilities
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    cross_validate,       # For cross-validation
    StratifiedKFold,      # For stratified splits (preserves class distribution)
)
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
    PrecisionRecallDisplay,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    make_scorer
)

# Imbalanced-learn: For SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline  # Special pipeline that works with SMOTE
from imblearn.over_sampling import SMOTE

# Explainable Boosting Machine
from interpret.glassbox import ExplainableBoostingClassifier

print("All libraries imported successfully!")

## Step 2: Load the Data

**What's happening here:**
- We load the pre-processed training and test data from parquet files
- `X_train` contains the features (input variables) for training
- `y_train` contains the target variable (what we want to predict) for training
- `X_test` and `y_test` are held out for final evaluation

**Why parquet?**
- Parquet is a columnar storage format that's faster and more efficient than CSV

In [ ]:
# The used file is the earlier version of WO provider; called incl. This includes the provider code and excludes the new variables like
# comorbidity count, oksfunctionscale, oks pain scale and oks daily life scale

# Define file paths
DATA_PATH = 'c:/Users/olifa/OneDrive/Documenten/GitHub/EAISI-Group-Project---NHS/code/EAISI-Group-Project---NHS/data/cleaned/'

# Load training data
X_train = pd.read_parquet(DATA_PATH + 'X_train_encoded_wo_provider_incl.parquet')
y_train = pd.read_parquet(DATA_PATH + 'y_train.parquet')

# Load test data (for final evaluation only!)
X_test = pd.read_parquet(DATA_PATH + 'X_test_encoded_wo_provider_incl.parquet')
y_test = pd.read_parquet(DATA_PATH + 'y_test.parquet')

# Convert y to 1D array (required by sklearn)
# The .values.ravel() converts from DataFrame to 1D numpy array
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"\nFeature names: {list(X_train.columns[:10])}... (showing first 10)")

In [ ]:
# Step 2b: Handle String Columns (One-Hot Encoding)
# StandardScaler requires numeric data, so we need to encode any string columns

from sklearn.preprocessing import OneHotEncoder

# Find string/object columns
string_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print("Data Type Summary:")
print(X_train.dtypes.value_counts())
print(f"\nString columns found: {string_cols}")

if string_cols:
    print(f"\nOne-hot encoding {len(string_cols)} string column(s)...")
    
    # Create encoder
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    
    # Fit on training data, transform both train and test
    encoded_train = encoder.fit_transform(X_train[string_cols])
    encoded_test = encoder.transform(X_test[string_cols])
    
    # Get new column names
    encoded_cols = encoder.get_feature_names_out(string_cols)
    print(f"Created {len(encoded_cols)} new encoded columns")
    
    # Drop original string columns
    X_train = X_train.drop(columns=string_cols)
    X_test = X_test.drop(columns=string_cols)
    
    # Add encoded columns
    X_train = pd.concat([
        X_train.reset_index(drop=True), 
        pd.DataFrame(encoded_train, columns=encoded_cols)
    ], axis=1)
    
    X_test = pd.concat([
        X_test.reset_index(drop=True), 
        pd.DataFrame(encoded_test, columns=encoded_cols)
    ], axis=1)
    
    print(f"\nNew shape - Train: {X_train.shape}, Test: {X_test.shape}")
else:
    print("\nNo string columns found - data is ready!")

print(f"\nAll columns are now numeric: {X_train.select_dtypes(include=['object']).columns.tolist() == []}")


## Step 3: Explore the Target Variable (Class Imbalance)

**What's happening here:**
- We check how many samples belong to each class (good outcome vs. poor outcome)
- **Class imbalance** occurs when one class has many more samples than the other
- This matters because models might just predict the majority class and still have high accuracy!

**Why this matters:**
- If 90% of patients have good outcomes, a model that always predicts "good" would be 90% accurate but useless
- We'll use techniques like class weights and SMOTE to address this

In [ ]:
# Count class distribution
unique, counts = np.unique(y_train, return_counts=True)
class_distribution = dict(zip(unique, counts))

print("Class Distribution in Training Set:")
print("=" * 40)
for cls, count in class_distribution.items():
    percentage = count / len(y_train) * 100
    label = "Poor Outcome" if cls == 1 else "Good Outcome"
    print(f"Class {cls} ({label}): {count} samples ({percentage:.1f}%)")

# Calculate imbalance ratio
majority_class = max(counts)
minority_class = min(counts)
imbalance_ratio = majority_class / minority_class
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Poor Outcome (1)', 'Good Outcome (0)'], counts, color=['#ff6b6b', '#4ecdc4'])
ax.set_ylabel('Number of Samples')
ax.set_title('Class Distribution in Training Data')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
            f'{count}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

## Step 4: Define the Models

**What's happening here:**
We create a dictionary of models, each with a different approach:

1. **EBM (Explainable Boosting Machine)**
   - A glass-box model that's both accurate AND interpretable
   - Uses gradient boosting on individual features

2. **Logistic Regression (No Weights)**
   - Basic baseline model
   - Treats all classes equally

3. **Logistic Regression (Balanced)**
   - Uses `class_weight='balanced'` to give more importance to minority class
   - Weight = n_samples / (n_classes * n_samples_for_class)

4. **Logistic Regression + SMOTE**
   - SMOTE = Synthetic Minority Over-sampling Technique
   - Creates synthetic samples of the minority class

5. **LASSO (L1 Regularization)**
   - Adds a penalty term that can shrink coefficients to exactly zero
   - Useful for feature selection (identifies important features)

In [ ]:
# Define models in a dictionary for easy iteration
# Each model is wrapped in a pipeline with StandardScaler for consistent preprocessing

from sklearn.ensemble import RandomForestClassifier

models = {
    # Model 1: Explainable Boosting Machine
    # - interactions=0 means no pairwise interactions (simpler model)
    # - random_state ensures reproducibility
    'EBM': Pipeline([
        ('scaler', StandardScaler()),  # Normalize features
        ('model', ExplainableBoostingClassifier(
            random_state=42,
            interactions=0,  # Can set to a number to include interactions
            n_jobs=-1        # Use all CPU cores
        ))
    ]),
    
    # Model 2: Logistic Regression - No class weights
    # - max_iter=1000 allows more iterations to converge
    'LogReg_NoWeights': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            random_state=42,
            max_iter=1000,
            class_weight=None  # No class weights - treats all classes equally
        ))
    ]),
    
    # Model 3: Logistic Regression - Balanced class weights
    # - Automatically adjusts weights inversely proportional to class frequencies
    'LogReg_Balanced': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            random_state=42,
            max_iter=1000,
            class_weight='balanced'  # Automatically balance classes
        ))
    ]),
    
    # Model 4: Logistic Regression + SMOTE
    # - Uses ImbPipeline from imblearn (special pipeline for resampling)
    # - SMOTE is only applied during training, not during prediction
    'LogReg_SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),  # Oversample minority class
        ('model', LogisticRegression(
            random_state=42,
            max_iter=1000
        ))
    ]),
    
    # Model 5: LASSO (L1 Regularized Logistic Regression)
    # - penalty='l1' adds L1 regularization
    # - solver='saga' is required for L1 penalty
    # - C=1.0 controls regularization strength (smaller = stronger regularization)
    'LASSO_LogReg': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            random_state=42,
            max_iter=1000,
            penalty='l1',          # L1 regularization (LASSO)
            solver='saga',         # Required for L1
            C=1.0,                 # Regularization strength
            class_weight='balanced'
        ))
    ]),
    
    # Model 6: Random Forest Classifier
    'Random_Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ])
}

print("Models defined:")
for name in models.keys():
    print(f"  - {name}")

## Step 5: Cross-Validation with Train/Validation Score Comparison

**What's happening here:**
- **Cross-validation** splits the training data into K folds
- The model trains on K-1 folds and validates on the remaining fold
- This is repeated K times, giving us K scores

**Why we compare train vs validation scores:**
- If **train score >> validation score**: Model is **overfitting** (memorizing training data)
- If **train score ≈ validation score**: Model generalizes well
- If both scores are low: Model is **underfitting** (too simple)

**Metrics we'll track:**
- **Precision**: Of all predicted positives, how many are actually positive?
- **Recall**: Of all actual positives, how many did we find?
- **F1**: Harmonic mean of precision and recall
- **ROC-AUC**: Area under the ROC curve

In [ ]:
# Define cross-validation strategy
# StratifiedKFold ensures each fold has the same class distribution as the full dataset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define scoring metrics
scoring = {
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# Store results
cv_results = {}

print("Running cross-validation for each model...")
print("=" * 60)

for name, model in models.items():
    print(f"\nTraining: {name}")
    
    # cross_validate returns both train and test scores when return_train_score=True
    scores = cross_validate(
        model, 
        X_train, 
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,  # Important: gives us train scores for overfitting detection
        n_jobs=1                 # Use all CPU cores
    )
    
    cv_results[name] = scores
    
    # Print summary
    print(f"  Validation Recall: {scores['test_recall'].mean():.3f} (+/- {scores['test_recall'].std():.3f})")
    print(f"  Validation Precision: {scores['test_precision'].mean():.3f} (+/- {scores['test_precision'].std():.3f})")

print("\n" + "=" * 60)
print("Cross-validation complete!")

## Step 6: Visualize Cross-Validation Results

**What's happening here:**
- We create a summary table comparing all models
- We visualize the results with bar charts
- We highlight any potential overfitting (large train-validation gap)

In [ ]:
# Create summary DataFrame
summary_data = []

for name, scores in cv_results.items():
    for metric in ['precision', 'recall', 'f1', 'roc_auc']:
        train_mean = scores[f'train_{metric}'].mean()
        train_std = scores[f'train_{metric}'].std()
        val_mean = scores[f'test_{metric}'].mean()
        val_std = scores[f'test_{metric}'].std()
        gap = train_mean - val_mean  # Overfitting indicator
        
        summary_data.append({
            'Model': name,
            'Metric': metric.upper(),
            'Train Mean': train_mean,
            'Train Std': train_std,
            'Val Mean': val_mean,
            'Val Std': val_std,
            'Gap (Train-Val)': gap
        })

summary_df = pd.DataFrame(summary_data)

# Display as pivot table for easy reading
print("\nModel Comparison Summary (Validation Scores):")
print("=" * 80)
pivot = summary_df.pivot_table(
    index='Model', 
    columns='Metric', 
    values='Val Mean'
).round(3)
print(pivot)

In [ ]:
# Visualize: Precision and Recall comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Get model names and metrics
model_names = list(cv_results.keys())

# Plot Precision
ax = axes[0]
train_precision = [cv_results[m]['train_precision'].mean() for m in model_names]
val_precision = [cv_results[m]['test_precision'].mean() for m in model_names]
val_precision_std = [cv_results[m]['test_precision'].std() for m in model_names]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, train_precision, width, label='Train', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, val_precision, width, label='Validation', color='#e74c3c', alpha=0.8,
               yerr=val_precision_std, capsize=3)

ax.set_xlabel('Model')
ax.set_ylabel('Precision')
ax.set_title('Precision: Train vs Validation')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

# Plot Recall
ax = axes[1]
train_recall = [cv_results[m]['train_recall'].mean() for m in model_names]
val_recall = [cv_results[m]['test_recall'].mean() for m in model_names]
val_recall_std = [cv_results[m]['test_recall'].std() for m in model_names]

bars1 = ax.bar(x - width/2, train_recall, width, label='Train', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, val_recall, width, label='Validation', color='#e74c3c', alpha=0.8,
               yerr=val_recall_std, capsize=3)

ax.set_xlabel('Model')
ax.set_ylabel('Recall')
ax.set_title('Recall: Train vs Validation')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Overfitting Analysis: Show the gap between train and validation scores
print("\nOverfitting Analysis (Train - Validation Gap):")
print("=" * 60)
print("A large positive gap indicates potential overfitting.")
print("Gaps > 0.05 are highlighted.\n")

gap_df = summary_df.pivot_table(
    index='Model',
    columns='Metric',
    values='Gap (Train-Val)'
).round(3)

# Style the dataframe to highlight concerning gaps
def highlight_overfitting(val):
    if val > 0.05:
        return 'background-color: #ffcccc'  # Light red
    elif val > 0.02:
        return 'background-color: #ffffcc'  # Light yellow
    return ''

styled_gap = gap_df.style.map(highlight_overfitting)

display(styled_gap)

## Step 7: Train Final Models and Get Feature Importances

**What's happening here:**
- We train each model on the full training set
- We extract feature importances (how much each feature contributes to predictions)

**Why feature importances matter:**
- Helps understand what the model is learning
- Validates that the model uses sensible features
- Can inform future feature engineering

In [ ]:
# Train all models on full training data
trained_models = {}

print("Training models on full training data...")
for name, model in models.items():
    print(f"  Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model

print("\nAll models trained!")

In [ ]:
# Extract feature importances
feature_names = X_train.columns.tolist()

def get_feature_importance(model, model_name, feature_names):
    if 'EBM' in model_name:
        ebm_model = model.named_steps['model']
        importances = ebm_model.term_importances()
        names = ebm_model.term_names_
        return pd.DataFrame({'Feature': names, 'Importance': importances})
    
    elif 'RandomForest' in model_name:
        rf_model = model.named_steps['model']
        importances = rf_model.feature_importances_
        return pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    
    else:
        lr_model = model.named_steps['model']
        importances = np.abs(lr_model.coef_[0])
        return pd.DataFrame({'Feature': feature_names, 'Importance': importances})


# LASSO-specific: Show which features have non-zero coefficients
# Only run this for LASSO model

if 'LASSO_LogReg' in trained_models:
    lasso_model = trained_models['LASSO_LogReg'].named_steps['model']
    
    if hasattr(lasso_model, 'coef_'):
        lasso_coef = lasso_model.coef_[0]
        
        n_nonzero = np.sum(lasso_coef != 0)
        n_zero = np.sum(lasso_coef == 0)
        
        print("LASSO Feature Selection Analysis:")
        print("=" * 50)
        print(f"Total features: {len(lasso_coef)}")
        print(f"Features with non-zero coefficients: {n_nonzero}")
        print(f"Features eliminated (zero coefficients): {n_zero}")
        print(f"\nLASSO kept {n_nonzero/len(lasso_coef)*100:.1f}% of features")
    else:
        print("LASSO model doesn't have coefficients available")

# Get importances for each model
# Make sure feature_names matches current X_train columns
feature_names = X_train.columns.tolist()

all_importances = {}
for name, model in trained_models.items():
    try:
        all_importances[name] = get_feature_importance(model, name, feature_names)
        print(f"✓ {name}: extracted {len(all_importances[name])} feature importances")
    except Exception as e:
        print(f"✗ {name}: Error - {e}")

print(f"\nSuccessfully extracted importances for {len(all_importances)} models")





In [ ]:
# Visualize top 15 features for each model
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, importance_df) in enumerate(all_importances.items()):
    if idx >= len(axes):
        break
        
    ax = axes[idx]
    
    # Get top 15 features
    top_features = importance_df.nlargest(15, 'Importance')
    
    # Create horizontal bar plot
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(top_features)))
    ax.barh(top_features['Feature'], top_features['Importance'], color=colors)
    ax.set_xlabel('Importance')
    ax.set_title(f'{name}\nTop 15 Features')
    ax.invert_yaxis()  # Highest importance at top

# Hide empty subplot if we have an odd number of models
if len(all_importances) < len(axes):
    axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# LASSO-specific: Show which features have non-zero coefficients
# (LASSO can shrink some coefficients to exactly zero)

lasso_model = trained_models['LASSO_LogReg'].named_steps['model']
lasso_coef = lasso_model.coef_[0]

n_nonzero = np.sum(lasso_coef != 0)
n_zero = np.sum(lasso_coef == 0)

print("LASSO Feature Selection Analysis:")
print("=" * 50)
print(f"Total features: {len(lasso_coef)}")
print(f"Features with non-zero coefficients: {n_nonzero}")
print(f"Features eliminated (zero coefficients): {n_zero}")
print(f"\nLASSO kept {n_nonzero/len(lasso_coef)*100:.1f}% of features")

## Step 8: Precision-Recall Curves

**What's happening here:**
- We plot the precision-recall curve for each model
- This shows the trade-off between precision and recall at different thresholds

**Why Precision-Recall curves instead of ROC curves?**
- For imbalanced datasets, PR curves are more informative
- A model with high AUC-PR is good at finding positives without too many false alarms

**How to read the curve:**
- Top-right corner is ideal (high precision AND high recall)
- The baseline is the proportion of positive class (random classifier)

In [ ]:
# Plot Precision-Recall curves on validation data using cross-validation
# We'll use the test set here for visualization since we trained on full training data

fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set2(np.linspace(0, 1, len(trained_models)))

for (name, model), color in zip(trained_models.items(), colors):
    # Get predicted probabilities
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        # For models without predict_proba, use decision function
        y_proba = model.decision_function(X_test)
    
    # Calculate precision-recall curve
    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
    
    # Calculate AUC-PR
    from sklearn.metrics import average_precision_score
    auc_pr = average_precision_score(y_test, y_proba)
    
    # Plot
    ax.plot(recall, precision, color=color, linewidth=2, label=f'{name} (AUC-PR={auc_pr:.3f})')

# Add baseline (random classifier)
baseline = y_test.mean()
ax.axhline(y=baseline, color='gray', linestyle='--', label=f'Baseline ({baseline:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves (Test Set)', fontsize=14)
ax.legend(loc='best', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot Recall at different thresholds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set2(np.linspace(0, 1, len(trained_models)))

for (name, model), color in zip(trained_models.items(), colors):
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
    
    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
    
    # Recall vs Threshold
    axes[0].plot(thresholds, recall[:-1], color=color, linewidth=2, label=name)
    
    # Precision vs Threshold
    axes[1].plot(thresholds, precision[:-1], color=color, linewidth=2, label=name)

axes[0].set_xlabel('Threshold', fontsize=12)
axes[0].set_ylabel('Recall', fontsize=12)
axes[0].set_title('Recall vs Classification Threshold', fontsize=14)
axes[0].legend(loc='best', fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Threshold', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision vs Classification Threshold', fontsize=14)
axes[1].legend(loc='best', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9: Final Test Set Evaluation

**What's happening here:**
- We evaluate all models on the held-out test set
- This gives us an unbiased estimate of how well each model will perform on new data

**Important:**
- The test set was NEVER used during training or cross-validation
- This is the final, honest evaluation of our models

In [ ]:
# Evaluate all models on test set
test_results = []

print("Final Test Set Evaluation")
print("=" * 70)

for name, model in trained_models.items():
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Get probabilities for ROC-AUC
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
    
    # Calculate metrics
    results = {
        'Model': name,
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }
    test_results.append(results)

# Create results DataFrame
test_results_df = pd.DataFrame(test_results).set_index('Model').round(3)
print(test_results_df)
print("\n" + "=" * 70)

In [ ]:
# Visualize test results
fig, ax = plt.subplots(figsize=(12, 6))

test_results_df.plot(kind='bar', ax=ax, width=0.8, colormap='viridis')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance on Test Set', fontsize=14)
ax.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_ylim(0, 1)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8, rotation=90, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    if idx >= len(axes):
        break
    
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    # Plot confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Poor', 'Good'])
    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
    axes[idx].set_title(f'{name}')

# Hide empty subplot
if len(trained_models) < len(axes):
    axes[-1].set_visible(False)

plt.suptitle('Confusion Matrices (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Step 10: Summary and Conclusions

**What we learned:**
- How different approaches handle class imbalance
- Which models show signs of overfitting
- Which features are most important for predicting outcomes
- The trade-off between precision and recall

In [ ]:
# Final summary
print("=" * 70)
print("SUMMARY")
print("=" * 70)

# Best model by each metric
for metric in ['Precision', 'Recall', 'F1', 'ROC-AUC']:
    best_model = test_results_df[metric].idxmax()
    best_score = test_results_df[metric].max()
    print(f"\nBest {metric}: {best_model} ({best_score:.3f})")

print("\n" + "=" * 70)
print("\nKey Observations:")
print("-" * 40)
print("1. Check the Train-Validation gap to identify overfitting")
print("2. SMOTE and balanced weights improve recall at cost of precision")
print("3. LASSO provides automatic feature selection")
print("4. EBM offers interpretable predictions with competitive performance")
print("\nChoose the model based on your priority:")
print("  - High Precision: Minimize false positives")
print("  - High Recall: Find all positive cases (important for medical screening)")
print("  - Balanced: Use F1 score")

## Bonus: EBM Interpretability

One of the key advantages of the Explainable Boosting Machine is its interpretability. Let's explore what the EBM learned.

In [ ]:
# EBM Global Explanation
from interpret import show

ebm_model = trained_models['EBM'].named_steps['model']

# Get global explanation
ebm_global = ebm_model.explain_global(name='EBM Global Importance')

# Show interactive plot (works in Jupyter)
show(ebm_global)

In [ ]:
# EBM Local Explanation (for a single prediction)
# Let's explain the first test sample

sample_idx = 0
sample = X_test.iloc[[sample_idx]]

ebm_local = ebm_model.explain_local(sample, y_test[[sample_idx]], name='EBM Local Explanation')
show(ebm_local)

---

## Next Steps

1. **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV to find optimal parameters
2. **Threshold Optimization**: Adjust the classification threshold based on your precision-recall preference
3. **Feature Engineering**: Create new features based on domain knowledge
4. **Ensemble Methods**: Combine multiple models for better performance

How to get 80% precision?

Cell 1: Find thresholds for 80% precision

In [ ]:
# Step 11: Adjust Thresholds for 80% Precision
# ============================================
# By default, classifiers use 0.5 as the threshold.
# We'll find the threshold where precision >= 0.8 for each model.

from sklearn.metrics import precision_recall_curve

target_precision = 0.80
optimal_thresholds = {}

print("Finding thresholds for 80% precision...")
print("=" * 70)

for name, model in trained_models.items():
    # Get predicted probabilities
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
    
    # Calculate precision at different thresholds
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    
    # Find threshold where precision >= target_precision
    # We want the LOWEST threshold that achieves target precision (maximizes recall)
    valid_indices = np.where(precisions >= target_precision)[0]
    
    if len(valid_indices) > 0:
        # Get the index with highest recall among valid precisions
        best_idx = valid_indices[0]  # First valid index has highest recall
        
        # Handle edge case where best_idx equals len(thresholds)
        if best_idx >= len(thresholds):
            best_idx = len(thresholds) - 1
            
        optimal_threshold = thresholds[best_idx]
        achieved_precision = precisions[best_idx]
        achieved_recall = recalls[best_idx]
        
        optimal_thresholds[name] = {
            'threshold': optimal_threshold,
            'precision': achieved_precision,
            'recall': achieved_recall
        }
        
        print(f"\n{name}:")
        print(f"  Optimal threshold: {optimal_threshold:.3f}")
        print(f"  Precision: {achieved_precision:.3f}")
        print(f"  Recall: {achieved_recall:.3f}")
    else:
        print(f"\n{name}:")
        print(f"  ⚠ Cannot achieve {target_precision:.0%} precision with this model")
        optimal_thresholds[name] = None

print("\n" + "=" * 70)


Cell 2: Evaluate with optimized thresholds

In [ ]:
# Step 12: Evaluate Models with Optimized Thresholds
# ==================================================

print("Test Set Results with 80% Precision Threshold")
print("=" * 70)

optimized_results = []

for name, model in trained_models.items():
    if optimal_thresholds[name] is None:
        print(f"\n{name}: Skipped (cannot achieve target precision)")
        continue
    
    threshold = optimal_thresholds[name]['threshold']
    
    # Get probabilities
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
    
    # Apply custom threshold
    y_pred_optimized = (y_proba >= threshold).astype(int)
    
    # Calculate metrics
    precision = precision_score(y_test, y_pred_optimized)
    recall = recall_score(y_test, y_pred_optimized)
    f1 = f1_score(y_test, y_pred_optimized)
    
    optimized_results.append({
        'Model': name,
        'Threshold': threshold,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    })

# Create results DataFrame
optimized_df = pd.DataFrame(optimized_results).set_index('Model').round(3)
print("\n")
print(optimized_df)
print("\n" + "=" * 70)


Cell 3: Compare default vs optimized thresholds

In [ ]:
# Step 13: Compare Default (0.5) vs Optimized Thresholds
# ======================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Get models that achieved target precision
valid_models = [m for m in trained_models.keys() if optimal_thresholds[m] is not None]

# Plot Precision comparison
ax = axes[0]
default_precision = [test_results_df.loc[m, 'Precision'] for m in valid_models]
optimized_precision = [optimized_df.loc[m, 'Precision'] for m in valid_models]

x = np.arange(len(valid_models))
width = 0.35

bars1 = ax.bar(x - width/2, default_precision, width, label='Default (0.5)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, optimized_precision, width, label='Optimized (≥80%)', color='#e74c3c', alpha=0.8)
ax.axhline(y=0.8, color='green', linestyle='--', linewidth=2, label='Target (80%)')

ax.set_xlabel('Model')
ax.set_ylabel('Precision')
ax.set_title('Precision: Default vs Optimized Threshold')
ax.set_xticks(x)
ax.set_xticklabels(valid_models, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

# Plot Recall comparison
ax = axes[1]
default_recall = [test_results_df.loc[m, 'Recall'] for m in valid_models]
optimized_recall = [optimized_df.loc[m, 'Recall'] for m in valid_models]

bars1 = ax.bar(x - width/2, default_recall, width, label='Default (0.5)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, optimized_recall, width, label='Optimized (≥80%)', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Recall')
ax.set_title('Recall: Default vs Optimized Threshold')
ax.set_xticks(x)
ax.set_xticklabels(valid_models, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("\n⚠ Note: Higher precision comes at the cost of lower recall!")


Cell 4: New Precision-Recall curve with thresholds marked

In [ ]:
# Step 14: Precision-Recall Curves with Optimal Thresholds Marked
# ===============================================================

fig, ax = plt.subplots(figsize=(12, 8))

colors = plt.cm.Set2(np.linspace(0, 1, len(trained_models)))

for (name, model), color in zip(trained_models.items(), colors):
    # Get predicted probabilities
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
    
    # Calculate precision-recall curve
    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
    
    # Calculate AUC-PR
    from sklearn.metrics import average_precision_score
    auc_pr = average_precision_score(y_test, y_proba)
    
    # Plot curve
    ax.plot(recall, precision, color=color, linewidth=2, label=f'{name} (AUC-PR={auc_pr:.3f})')
    
    # Mark the optimal threshold point
    if optimal_thresholds[name] is not None:
        opt_recall = optimal_thresholds[name]['recall']
        opt_precision = optimal_thresholds[name]['precision']
        ax.scatter([opt_recall], [opt_precision], color=color, s=150, marker='*', 
                   edgecolors='black', linewidths=1, zorder=5)

# Add 80% precision line
ax.axhline(y=0.80, color='red', linestyle='--', linewidth=2, label='Target Precision (80%)')

# Add baseline
baseline = y_test.mean()
ax.axhline(y=baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves with Optimal Threshold Points (★)', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("★ markers show the operating point for each model at 80% precision")


This outcome reveals an important insight about the models:

What the numbers mean:
Threshold: The cutoff probability needed to achieve 80% precision (instead of default 0.5)
Precision: Of all patients predicted as "good outcome", 80% actually had good outcomes
Recall: Of all patients who actually had good outcomes, how many did we find
F1: Balance between precision and recall

The Problem:
Your recall is extremely low (2-7%). This means:
- EBM (best): Only finds 7.1% of patients with good outcomes
- LogReg_SMOTE (worst): Only finds 2.3% of patients with good outcomes

In Plain English:
Imagine 100 patients who will have a good outcome after surgery:
- With EBM at 80% precision: You'd correctly identify only 7 of them
- The other 93 patients would be missed (predicted as poor outcome)

Why This Happens:
- Your classes likely overlap a lot - The features don't strongly separate good from poor outcomes
- Trade-off is unavoidable - To be 80% sure someone will have a good outcome, you have to be very conservative, missing most positive cases

What This Tells You:
If your goal is...	
- Not missing patients (high recall): Use lower threshold, accept lower precision
- Being confident in predictions (high precision): Current setup, but you'll miss most cases
- Balanced approach: Use default 0.5 threshold (check your earlier results)
Visualization:

High Precision (80%) = Very few predictions, but mostly correct
                       ████░░░░░░░░░░░░░░░░ (finds ~5% of good outcomes)

High Recall (80%)    = Many predictions, but includes mistakes  
                       ████████████████░░░░ (finds ~80% of good outcomes)
